# Road Defect Detection — trial inference on Colab

Run the pipeline on a real dashcam or drone video with GPU acceleration, and get back:

* **`annotated.mp4`** — road outline, hatched unassessable areas, defect masks, track IDs, live count
* **`defects.csv`** — one row per unique defect: class, IRC severity band, area in m², GPS, whether it was measurable
* **`segments.csv`** — per-100 m condition grade with coverage
* **`report.html`** — headline counts, unassessable %, assessment zones, sample crops
* **`summary.json`** — every per-stage number the run produced

---

### Two things to know before you start

**1. There are no fine-tuned weights in this project yet.** The only checkpoint available
is stock COCO, which knows about people and cars, not potholes. If you run without
supplying weights, the pipeline will execute end-to-end and every geometric/label-free
stage will work — but the *detector* output will be meaningless and classes will appear
as `UNMAPPED_CLASS_*`. Step 3 has a slot for real weights the moment you have them.

**2. Seven distress types does not mean seven models.** One segmentation model covers
potholes and cracks. The crack sub-types (longitudinal / transverse / alligator) are
derived by measuring orientation on the road plane, not by the network. Edge damage,
ravelling, rutting and drainage need **no model at all** — they are measured
geometrically or statistically. So you only ever supply one weights file.

---

**Before running:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*.

## 1 · Check the GPU

In [ ]:
import subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU DETECTED")
print()
try:
    import torch
    print(f"torch {torch.__version__}   cuda available: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        print("\n  Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session")
except ImportError:
    print("torch not yet installed - the next cells will handle it")

## 2 · Get the pipeline code

The repository is **private**, so cloning needs a GitHub token.

**Recommended:** add a token to Colab Secrets (the 🔑 icon in the left sidebar) named
`GITHUB_TOKEN`, with `repo` scope. Create one at
*GitHub → Settings → Developer settings → Personal access tokens*.

If you would rather not use a token, set `USE_ZIP = True` below and upload a zip of the
repo instead.

In [ ]:
#@title Clone the repository { display-mode: "form" }
GITHUB_USER = "Prashant2804"  #@param {type:"string"}
REPO        = "road-defect-pipeline"  #@param {type:"string"}
BRANCH      = "master"  #@param {type:"string"}
USE_ZIP     = False  #@param {type:"boolean"}

import os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path("/content") / REPO

if USE_ZIP:
    from google.colab import files
    print("Upload a zip of the repository...")
    up = files.upload()
    name = next(iter(up))
    if ROOT.exists():
        shutil.rmtree(ROOT)
    shutil.unpack_archive(name, "/content")
    # A GitHub zip unpacks as <repo>-<branch>/; normalise the folder name.
    if not ROOT.exists():
        cand = [p for p in Path("/content").iterdir()
                if p.is_dir() and p.name.startswith(REPO)]
        if cand:
            cand[0].rename(ROOT)
else:
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if not token:
        import getpass
        print("No GITHUB_TOKEN in Colab Secrets.")
        token = getpass.getpass("Paste a GitHub token (input hidden, not stored): ").strip()

    if ROOT.exists():
        shutil.rmtree(ROOT)
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(ROOT)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Never echo the token back in an error message.
        print(r.stderr.replace(token, "***") if token else r.stderr)
        raise SystemExit("Clone failed - check the token has 'repo' scope and the branch exists")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print(f"\nRepository at {ROOT}")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3 · Install dependencies

Colab already has a CUDA build of PyTorch. The cell below installs the rest and then
**verifies CUDA still works** — installing `ultralytics` can occasionally pull a CPU
build of torch over the top, which would silently make everything 20× slower.

In [ ]:
import subprocess, sys, json

# Torch is deliberately NOT reinstalled: Colab's build is already CUDA-enabled and
# matched to the driver.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics>=8.3.0", "lap>=0.5.12", "supervision>=0.25.0",
                "gpxpy>=1.6", "jinja2>=3.1", "reportlab>=4.1", "pyyaml>=6.0"],
               check=True)

# Check torch in a FRESH interpreter rather than reloading it here.
# importlib.reload(torch) re-runs torch's module init, which re-registers its C++
# operator namespaces and fails with "Only a single TORCH_LIBRARY can be used to
# register the namespace triton". It also would not answer the question: a
# C-extension already loaded into this process cannot be swapped by reload, so only
# a new interpreter sees what pip actually left on disk.
probe = subprocess.run(
    [sys.executable, "-c",
     "import json, torch; "
     "print(json.dumps({'v': torch.__version__, 'cuda': torch.cuda.is_available()}))"],
    capture_output=True, text=True)

if probe.returncode != 0:
    print("Could not import torch in a fresh interpreter:\n")
    print(probe.stderr[-1500:])
    raise SystemExit("torch is broken - Runtime -> Restart session, then re-run cell 2 onwards")

info = json.loads(probe.stdout.strip().splitlines()[-1])
print(f"on disk: torch {info['v']}   cuda={info['cuda']}")

# If this kernel already had a different torch imported, the in-memory copy is stale
# and every later cell would keep using it.
stale = "torch" in sys.modules and sys.modules["torch"].__version__ != info["v"]
if stale:
    print(f"in memory: torch {sys.modules['torch'].__version__}  <-- STALE")
    print("\n  pip replaced torch. Runtime -> Restart session, then re-run from cell 2.")
elif not info["cuda"]:
    print("\n  CUDA is not available. Two possible causes:")
    print("    1. The runtime has no GPU  -> Runtime -> Change runtime type -> T4 GPU")
    print("    2. A CPU build of torch got installed -> run the cell below, then restart")
    print("\n  subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',")
    print("                  '--force-reinstall', 'torch', 'torchvision',")
    print("                  '--index-url', 'https://download.pytorch.org/whl/cu121'])")
else:
    print("GPU inference is ready.")

In [ ]:
# Health check: versions, ffmpeg's v360 filter, every pipeline stage, and what this
# camera configuration can actually resolve.
!python tools/doctor.py

## 4 · Load your video from Google Drive

Three ways, pick one with `SOURCE`:

| `SOURCE` | Use when |
|---|---|
| `drive_link` | You have a shareable link. **Set the file to "Anyone with the link"** |
| `drive_mount` | The file is in your own Drive — you'll be asked to authorise |
| `upload` | Small file straight from your machine |

A shared *link* works for a single file. For a whole folder of clips, `drive_mount` is
easier.

In [ ]:
#@title Fetch the video { display-mode: "form" }
SOURCE      = "drive_link"  #@param ["drive_link", "drive_mount", "upload"]
DRIVE_LINK  = ""  #@param {type:"string"}
DRIVE_PATH  = "/content/drive/MyDrive/road_videos/road.mp4"  #@param {type:"string"}

import re, shutil, subprocess, sys
from pathlib import Path

VIDEO_DIR = Path("/content/videos"); VIDEO_DIR.mkdir(exist_ok=True)
VIDEO = None

if SOURCE == "drive_link":
    if not DRIVE_LINK.strip():
        raise SystemExit("Paste a Google Drive share link into DRIVE_LINK")
    # Accept any of the usual Drive URL shapes, or a bare file id.
    m = re.search(r"/d/([A-Za-z0-9_-]{20,})", DRIVE_LINK) or \
        re.search(r"[?&]id=([A-Za-z0-9_-]{20,})", DRIVE_LINK)
    file_id = m.group(1) if m else DRIVE_LINK.strip()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    out = str(VIDEO_DIR / "input.mp4")
    got = gdown.download(id=file_id, output=out, quiet=False)
    if not got:
        raise SystemExit(
            "Download failed. The most common cause is sharing: open the file in Drive, "
            "Share -> General access -> 'Anyone with the link'. Very large files may also "
            "hit Drive's virus-scan interstitial - use drive_mount instead.")
    VIDEO = Path(out)

elif SOURCE == "drive_mount":
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = Path(DRIVE_PATH)
    if not VIDEO.exists():
        raise SystemExit(f"Not found: {VIDEO}\nCheck the path in the file browser on the left.")

else:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    VIDEO = VIDEO_DIR / name
    shutil.move(name, VIDEO)

# Probe it, because a wrong resolution or frame rate changes what is resolvable.
import json
p = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                    "-show_entries", "stream=width,height,r_frame_rate,nb_frames,codec_name",
                    "-show_entries", "format=duration,size", "-of", "json", str(VIDEO)],
                   capture_output=True, text=True)
info = json.loads(p.stdout or "{}")
st = (info.get("streams") or [{}])[0]
fmt = info.get("format", {})
num, _, den = (st.get("r_frame_rate") or "0/1").partition("/")
fps = float(num) / float(den or 1)
print(f"\n  {VIDEO}")
print(f"  {st.get('width')}x{st.get('height')}  {fps:.1f} fps  "
      f"{float(fmt.get('duration', 0)):.1f}s  "
      f"{int(fmt.get('size', 0)) / 1e6:.0f} MB  {st.get('codec_name')}")

## 5 · Load model weights

**If you have fine-tuned weights**, point at them: a Drive link, a direct URL, or a
path in your mounted Drive.

**If you do not**, leave `WEIGHTS_SOURCE = "none"`. The pipeline still runs end to end
and everything geometric still works — road segmentation, water/mud occlusion, edge
damage, ravelling, drainage, assessment zones, the validity gate, the report. Only the
*detector* output will be meaningless, and it will be labelled as such.

Where fine-tuned weights come from:
* your own `rdd train` run (`out/<name>/train/weights/best.pt`)
* a Roboflow Universe road-damage model, exported as YOLOv8/v11
* an RDD2022-trained checkpoint

The classes in the checkpoint must match `model.classes` in `config.yaml`, or the run
will warn loudly and the labels will be wrong. The next cell checks this for you.

In [ ]:
#@title Fetch weights { display-mode: "form" }
WEIGHTS_SOURCE = "rdd2022_hf"  #@param ["rdd2022_hf", "none", "drive_link", "drive_path", "url", "upload"]
WEIGHTS_LINK   = ""  #@param {type:"string"}
WEIGHTS_PATH   = "/content/drive/MyDrive/weights/best.pt"  #@param {type:"string"}

import re, shutil, subprocess, sys
from pathlib import Path

W_DIR = Path("/content/weights"); W_DIR.mkdir(exist_ok=True)
WEIGHTS, CLASS_MAP = None, {}

if WEIGHTS_SOURCE == "rdd2022_hf":
    # yolo12s fine-tuned on RDD2022 (MIT licence). Its five classes are the RDD
    # damage codes, which cover four of our nine categories. Detection only, no
    # masks - see the note below for what that costs.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
                   check=True)
    from huggingface_hub import hf_hub_download
    WEIGHTS = Path(hf_hub_download("rezzzq/yolo12s-road-damage-rdd2022",
                                   "yolo12s_RDD2022_best.pt"))
    CLASS_MAP = {"D00": "longitudinal_crack", "D10": "transverse_crack",
                 "D20": "alligator_crack", "D40": "pothole",
                 "Repair": None}     # a past repair is not a defect

elif WEIGHTS_SOURCE == "drive_link":
    m = re.search(r"/d/([A-Za-z0-9_-]{20,})", WEIGHTS_LINK) or \
        re.search(r"[?&]id=([A-Za-z0-9_-]{20,})", WEIGHTS_LINK)
    fid = m.group(1) if m else WEIGHTS_LINK.strip()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    WEIGHTS = Path(gdown.download(id=fid, output=str(W_DIR / "best.pt"), quiet=False))
elif WEIGHTS_SOURCE == "drive_path":
    WEIGHTS = Path(WEIGHTS_PATH)
    if not WEIGHTS.exists():
        raise SystemExit(f"Not found: {WEIGHTS} (mount Drive in the previous cell first)")
elif WEIGHTS_SOURCE == "url":
    WEIGHTS = W_DIR / "best.pt"
    subprocess.run(["wget", "-q", "-O", str(WEIGHTS), WEIGHTS_LINK], check=True)
elif WEIGHTS_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    WEIGHTS = W_DIR / name
    shutil.move(name, WEIGHTS)

if WEIGHTS is None:
    print("  No weights supplied.")
    print("  The pipeline runs on the stock COCO warm-start, so DETECTION OUTPUT WILL")
    print("  NOT BE MEANINGFUL - classes appear as UNMAPPED_CLASS_*. Everything")
    print("  geometric and label-free still works and is worth looking at.")
else:
    sys.path.insert(0, "src")
    from ultralytics import YOLO
    _m = YOLO(str(WEIGHTS))
    names = [_m.names[k] for k in sorted(_m.names)] if isinstance(_m.names, dict) else list(_m.names)
    print(f"  {WEIGHTS.name}   {len(names)} classes: {names}")

    if CLASS_MAP:
        print("\n  class map (checkpoint name -> pipeline class):")
        for k in names:
            v = CLASS_MAP.get(k, k)
            print(f"    {k:<8} -> {v if v else '(dropped)'}")
        print("\n  Without this map, ids resolve POSITIONALLY: D00 (a longitudinal")
        print("  crack) would be reported as 'pothole' simply because both sit at")
        print("  index 0 - silently, with no error.")
    else:
        print("\n  No class map set. If this checkpoint's classes are not in the same")
        print("  ORDER as model.classes in config.yaml, every label will be wrong.")
        print("  Set CLASS_MAP above if they differ.")

    # A detection model gives boxes, not masks. The pipeline copes, but area is then
    # the box area rather than the defect's, and alligator cracking cannot be
    # confirmed by counting enclosed cells - so the model's own D20 label is trusted.
    task = getattr(_m, "task", "detect")
    if task != "segment":
        print(f"\n  NOTE: this is a '{task}' (box) model, not segmentation.")
        print("  Defect AREA will be the bounding box, which overestimates - especially")
        print("  for cracks, where a thin diagonal fills little of its box. Crack")
        print("  orientation still works; alligator relies on the model's own label.")

## 6 · Camera geometry — the part that matters most

These four numbers convert pixels to metres. Get them wrong and every area, crack width
and severity band is wrong by the same factor, silently.

* **`CAMERA_HEIGHT_M`** — measure it. Ground to lens. This scales *everything*.
* **`CAMERA_HFOV_DEG`** — *horizontal* field of view. Dashcam boxes usually quote the
  **diagonal** figure ("170°"), which is much larger. A typical horizontal FOV is
  90–120°. If unsure, 100° is a better guess than the number on the box.
* **`VIEW`** — `car_flat` for a normal dashcam, `car_360` for equirectangular 360
  footage, `drone_nadir` for straight-down drone video.
* **`CAMERA_PITCH_DEG`** — downward tilt. Only a starting value: the pipeline
  re-estimates pitch and yaw per clip from the vanishing point.

The cell prints the **assessment zones** that result: the range over which each class
can actually be resolved. Cracks needing 5 mm/px are typically only assessable a couple
of metres ahead — that is a sensor limit, not a model limit.

In [ ]:
#@title Camera setup { display-mode: "form" }
VIEW             = "car_flat"  #@param ["car_flat", "car_360", "drone_nadir"]
CAMERA_HEIGHT_M  = 1.35  #@param {type:"number"}
CAMERA_HFOV_DEG  = 100.0  #@param {type:"number"}
CAMERA_PITCH_DEG = 8.0  #@param {type:"number"}
RUN_NAME         = "colab"  #@param {type:"string"}

import sys
sys.path.insert(0, "src")

# No importlib.reload here either: reloading rdd.config would create a second Cfg
# class, and the isinstance checks inside it would stop recognising objects built by
# the first one. In a linear notebook run there is nothing to reload anyway.
from rdd.config import load_config
from rdd.geometry.calibration import build_camera
from rdd.geometry.zones import build_zones

cfg = load_config("config.yaml")
cfg.set_path("view.profile", VIEW)
cfg.set_path("geometry.camera.height_m", CAMERA_HEIGHT_M)
cfg.set_path("geometry.camera.h_fov_deg", CAMERA_HFOV_DEG)
cfg.set_path("geometry.camera.pitch_deg", CAMERA_PITCH_DEG)

import cv2
cap = cv2.VideoCapture(str(VIDEO))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

cam = build_camera(cfg, W, H)
zones = build_zones(cfg, cam)
near, far = cam.visible_range()

print(f"\n  {W}x{H}, ground visible from {near:.1f} m to {far:.0f} m ahead\n")
print(f"  {'class':<22}{'assessable range':<22}{'needs'}")
print("  " + "-" * 58)
for name, z in sorted(zones.zones.items()):
    if z.achievable:
        rng = f"{z.z_near_m:.1f} - {z.z_far_m:.1f} m"
    else:
        rng = "NOT ACHIEVABLE"
    print(f"  {name:<22}{rng:<22}{1000 * z.required_gsd_m:.0f} mm/px")

bad = zones.unachievable()
if bad:
    print(f"\n  Cannot be assessed with this camera: {', '.join(bad)}")
    print("  A narrower field of view extends these ranges; a higher mount does not.")

## 7 · Check the road mask before trusting anything downstream

Every later number is computed *inside* the road mask, so if the outline is wrong,
nothing after it can be right. This takes a few seconds and is worth doing every time
you point the pipeline at new footage.

**Green outline** = detected road · **hatched blue** = water · **hatched brown** = mud

In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

#@markdown Preview on a short slice - a full-length clip is not needed to judge the mask.
PREVIEW_SECONDS = 20  #@param {type:"integer"}

preview_src = VIDEO
if PREVIEW_SECONDS and PREVIEW_SECONDS > 0:
    preview_src = Path("/content/videos/preview_slice.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-t", str(PREVIEW_SECONDS),
                    "-i", str(VIDEO), "-c", "copy", str(preview_src)], check=True)
    print(f"Using the first {PREVIEW_SECONDS}s for the preview\n")

cmd = ["python", "run.py", "roadseg", "--input", str(preview_src), "--view", VIEW,
       "--n", "4",
       "--set", f"geometry.camera.height_m={CAMERA_HEIGHT_M}",
       "--set", f"geometry.camera.h_fov_deg={CAMERA_HFOV_DEG}",
       "--set", f"geometry.camera.pitch_deg={CAMERA_PITCH_DEG}"]
print(" ".join(cmd), "\n" + "-" * 70)

# Streamed, not captured. With capture_output the cell shows nothing until it
# finishes, which on a long clip is indistinguishable from a hang.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 70)

for pth in sorted(Path("out/roadseg_preview").glob("*.jpg"))[:4]:
    display(Image(filename=str(pth), width=760))

**If the outline is wrong**, adjust before running the full pipeline — it is far cheaper
to fix here:

```python
# road sits lower/higher in frame than assumed
--set view.road_prior.top_y=0.62
# road appears narrower or wider
--set view.road_prior.bottom_half_width=0.40
# segmentation too strict (mask too small) or too loose (leaks into the verge)
--set roadseg.classical.distance_tau=3.0
```

## 8 · Run the pipeline on GPU

Everything happens here: quality assessment, camera calibration from the vanishing
point, road segmentation, water/mud occlusion, the validity gate, detection + tracking
gated to the road, crack classification by ground geometry, edge/ravelling/drainage
measurement, IRC severity, and the report.

Expect roughly real-time to 3× real-time on a T4, depending on resolution.

In [ ]:
#@title Inference settings { display-mode: "form" }
#@markdown **PRESET is the speed lever.** The GPU is not the bottleneck here - the
#@markdown per-frame CPU work (road segmentation, surface analysis, optical flow) is.
#@markdown So the way to go faster is to do that work on fewer frames, NOT to raise
#@markdown IMGSZ, which only makes each frame more expensive.
#@markdown - `fast` detects every 3rd frame (~2.5x). At 30 fps the vehicle moves ~28 cm
#@markdown   per frame, so consecutive frames re-inspect the same tarmac.
#@markdown - `turbo` every 8th - a first look at long footage.
#@markdown - `accurate` every frame.
#@markdown
#@markdown Skipped frames are still written to the annotated video, so it stays a
#@markdown complete record. Set MAX_SECONDS to trial-run on a short slice first.
PRESET      = "fast"  #@param ["fast", "turbo", "accurate", "none"]
CONF        = 0.25  #@param {type:"number"}
IMGSZ       = 960  #@param {type:"integer"}
MAX_SECONDS = 60  #@param {type:"integer"}

import subprocess, time
from pathlib import Path

src = VIDEO
# Trim first if asked: a short clip is the sane way to sanity-check settings before
# committing to a long survey.
if MAX_SECONDS and MAX_SECONDS > 0:
    src = Path("/content/videos/trimmed.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-t", str(MAX_SECONDS),
                    "-i", str(VIDEO), "-c", "copy", str(src)], check=True)
    print(f"Trimmed to the first {MAX_SECONDS}s -> {src}")

cmd = ["python", "run.py", "--input", str(src), "--view", VIEW,
       "--device", "cuda", "--output", "out",
       "--set", f"run.name={RUN_NAME}",
       "--set", f"inference.conf={CONF}",
       "--set", f"inference.imgsz={IMGSZ}",
       "--set", f"geometry.camera.height_m={CAMERA_HEIGHT_M}",
       "--set", f"geometry.camera.h_fov_deg={CAMERA_HFOV_DEG}",
       "--set", f"geometry.camera.pitch_deg={CAMERA_PITCH_DEG}"]
if PRESET and PRESET != "none":
    cmd += ["--preset", PRESET]
if WEIGHTS:
    cmd += ["--set", f"inference.weights={WEIGHTS}"]
if CLASS_MAP:
    # Passed as JSON so the loader resolves detections by the checkpoint's own class
    # names rather than positionally.
    import json as _json
    cmd += ["--set", f"model.class_map={_json.dumps(CLASS_MAP)}"]

print(" ".join(cmd), "\n" + "=" * 70)
t0 = time.time()
# Stream the log rather than buffering: a long run with no output looks hung.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("=" * 70)
print(f"Finished in {time.time() - t0:.0f}s with exit code {proc.returncode}")

OUT = Path("out") / RUN_NAME
print(f"Artifacts: {sorted(p.name for p in OUT.glob('*'))}")

## 9 · Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

OUT = Path("out") / RUN_NAME
if not (OUT / "summary.json").exists():
    raise SystemExit(
        f"No results in {OUT}. The previous cell did not finish — scroll up to its
"
        f"last lines for the reason. Only annotated.mp4 and run.log are written before
"
        f"the report stage, so seeing just those two means it stopped during inference.")
summary = json.loads((OUT / "summary.json").read_text())
pipe = summary["pipeline"]
val = pipe["validity"]
surf = pipe["surface"]

print("  DETECTION")
print(f"    unique defects        {summary['unique_total']}")
print(f"    measurable            {summary['assessable_total']}")
print(f"    indeterminate         {summary['indeterminate_total']}  (hidden under water/mud)")
print(f"    per class             {summary['unique_counts_per_class']}")
print()
print("  COVERAGE - what these numbers actually apply to")
print(f"    frames assessed       {val['frame_coverage']:.1%}  ({val['frames_assessable']}/{val['frames']})")
if val.get("blocked_by_gate"):
    print(f"    excluded because      {val['blocked_by_gate']}")
print(f"    road unassessable     {surf['road_surface_unassessable_frac']:.1%}"
      f"  (water {surf['road_surface_water_frac']:.1%}, mud {surf['road_surface_mud_frac']:.1%})")
print()
print("  METHOD")
print(f"    severity basis        {summary.get('severity', {}).get('basis')}")
print(f"    ground scale          {pipe['ground_scale']}")
print(f"    rejected off-road     {pipe['detections_rejected_off_road']}")
print(f"    rejected out of zone  {pipe['detections_rejected_out_of_zone']}")
print(f"    rejected as confusers {pipe['confuser_rejections']['by_confuser']}")
print()
print("  SURFACE CONDITIONS (label-free)")
for k, v in pipe["surface_conditions"].items():
    headline = {kk: vv for kk, vv in v.items() if kk not in ("basis", "frames_measured")}
    print(f"    {k:<14}{headline}")

In [ ]:
# Per-defect table. `assessable=no` means detected but hidden under water/mud, so
# deliberately not severity-scored.
import pandas as pd
df = pd.read_csv(OUT / "defects.csv")
print(f"{len(df)} unique defects\n")
cols = [c for c in ["track_id", "class", "first_t_s", "area_m2", "irc_level",
                    "severity_level", "assessable", "occluded_frac", "peak_conf",
                    "lat", "lon"] if c in df.columns]
display(df[cols].head(40))

In [ ]:
# Per-100 m condition grades. A segment below the coverage floor is graded
# "indeterminate" rather than "sound" - a stretch nobody could see is not a good stretch.
seg = OUT / "segments.csv"
if seg.exists():
    display(pd.read_csv(seg).head(30))
else:
    print("No segments.csv (report.segments.enabled is false)")

### Annotated video

Re-encoded smaller for inline playback. The full-quality original is in the download
below.

In [ ]:
import base64, subprocess
from pathlib import Path
from IPython.display import HTML, display

src = OUT / "annotated.mp4"
preview = Path("/content/preview.mp4")
# Colab embeds the file as base64 in the page, so a full-resolution survey video would
# make the notebook unusable. 720p at CRF 30 is plenty to check the overlays.
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
                "-vf", "scale='min(1280,iw)':-2", "-c:v", "libx264", "-crf", "30",
                "-preset", "veryfast", "-pix_fmt", "yuv420p", str(preview)], check=True)

mb = preview.stat().st_size / 1e6
if mb > 60:
    print(f"Preview is {mb:.0f} MB - too large to embed. Download it below instead.")
else:
    b64 = base64.b64encode(preview.read_bytes()).decode()
    display(HTML(f'<video width="900" controls><source src="data:video/mp4;base64,{b64}" '
                 f'type="video/mp4"></video>'))
    print("Green outline = road · hatched = unassessable (water/mud) · "
          "banner = frame not assessed")

In [ ]:
# The HTML report, inline. Includes the unassessable-% banner, assessment zones,
# severity basis, chainage grades and sample crops.
from IPython.display import HTML
HTML((OUT / "report.html").read_text(encoding="utf-8"))

## 10 · Save the results

Zip everything and either download it or write it back to Drive.

In [ ]:
#@title Export { display-mode: "form" }
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_OUT_DIR = "/content/drive/MyDrive/road_defect_results"  #@param {type:"string"}

import shutil
from pathlib import Path

archive = shutil.make_archive(f"/content/{RUN_NAME}_results", "zip", root_dir=str(OUT))
size = Path(archive).stat().st_size / 1e6
print(f"{archive}  ({size:.1f} MB)")
print("  annotated.mp4, defects.csv, segments.csv, report.html, summary.json, "
      "manifest.json, crops/, run.log")

if SAVE_TO_DRIVE:
    from google.colab import drive
    import os
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    dest = Path(DRIVE_OUT_DIR); dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, dest)
    shutil.copy(OUT / "annotated.mp4", dest / f"{RUN_NAME}_annotated.mp4")
    print(f"Copied to {dest}")
else:
    from google.colab import files
    files.download(archive)

---

## What to do with this

**Reading the output honestly.** Two numbers qualify everything else:

* **frames assessed** — defect counts apply only to this fraction of the route.
  Excluded frames were never inspected; they are not evidence of intact road.
* **road unassessable** — the share of visible road surface hidden under water or mud.
  If this is high, a low defect count is a statement about visibility, not condition.

**Severity basis** tells you whether numbers are physical. `absolute_m2` means real
measurements against IRC bands. `relative_px` means no ground scale was available, so
severity is normalised within this clip only — the biggest defect present is always
"high", and two videos are not comparable. Supplying camera height and FOV in step 6 is
what moves you from the second to the first.

**Next steps, in order of value:**

1. **Label a validation set** — ~35 instances per class, minimum. That is a hard
   statistical floor: at 100% observed precision the confidence interval's lower bound
   only clears 90% at n=35, so a smaller set cannot certify the target however good the
   detector is.
2. **Measure precision**: `python run.py evaluate --defects out/<name>/defects.csv
   --truth gt.csv` — produces per-class thresholds and a certification table splitting
   certified from indicative.
3. **Fine-tune** only on the classes that missed the target. Measuring first means
   labelling effort goes where it is needed instead of being spread evenly.

**If crack classes show as NOT ACHIEVABLE in step 6**, the camera cannot resolve them at
any range. Narrowing the field of view is the effective fix; raising the mount is
counter-productive because it pushes the nearest visible ground away faster than it
improves resolution.